<div class='alert alert-warning'>

NumPy's interactive examples are experimental and may not always work as expected, with high load times especially on low-resource platforms, and the version of NumPy might not be in sync with the one you are browsing the documentation for. If you encounter any issues, please report them on the [NumPy issue tracker](https://github.com/numpy/numpy/issues).

</div>

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
x = np.arange(6)
x.shape

(6,)

In [ ]:
v = sliding_window_view(x, 3)
v.shape

(4, 3)

In [ ]:
v

array([[0, 1, 2],
       [1, 2, 3],
       [2, 3, 4],
       [3, 4, 5]])

This also works in more dimensions, e.g.


In [ ]:
i, j = np.ogrid[:3, :4]
x = 10*i + j
x.shape

(3, 4)

In [ ]:
x

array([[ 0,  1,  2,  3],
       [10, 11, 12, 13],
       [20, 21, 22, 23]])

In [ ]:
shape = (2,2)
v = sliding_window_view(x, shape)
v.shape

(2, 3, 2, 2)

In [ ]:
v

array([[[[ 0,  1],
         [10, 11]],
        [[ 1,  2],
         [11, 12]],
        [[ 2,  3],
         [12, 13]]],
       [[[10, 11],
         [20, 21]],
        [[11, 12],
         [21, 22]],
        [[12, 13],
         [22, 23]]]])

The axis can be specified explicitly:


In [ ]:
v = sliding_window_view(x, 3, 0)
v.shape

(1, 4, 3)

In [ ]:
v

array([[[ 0, 10, 20],
        [ 1, 11, 21],
        [ 2, 12, 22],
        [ 3, 13, 23]]])

The same axis can be used several times. In that case, every use reduces
the corresponding original dimension:


In [ ]:
v = sliding_window_view(x, (2, 3), (1, 1))
v.shape

(3, 1, 2, 3)

In [ ]:
v

array([[[[ 0,  1,  2],
         [ 1,  2,  3]]],
       [[[10, 11, 12],
         [11, 12, 13]]],
       [[[20, 21, 22],
         [21, 22, 23]]]])

Combining with stepped slicing (`::step`), this can be used to take sliding
views which skip elements:


In [ ]:
x = np.arange(7)
sliding_window_view(x, 5)[:, ::2]

array([[0, 2, 4],
       [1, 3, 5],
       [2, 4, 6]])

or views which move by multiple elements


In [ ]:
x = np.arange(7)
sliding_window_view(x, 3)[::2, :]

array([[0, 1, 2],
       [2, 3, 4],
       [4, 5, 6]])

A common application of `sliding_window_view` is the calculation of running
statistics. The simplest example is the
[moving average](https://en.wikipedia.org/wiki/Moving_average):


In [ ]:
x = np.arange(6)
x.shape

(6,)

In [ ]:
v = sliding_window_view(x, 3)
v.shape

(4, 3)

In [ ]:
v

array([[0, 1, 2],
       [1, 2, 3],
       [2, 3, 4],
       [3, 4, 5]])

In [ ]:
moving_average = v.mean(axis=-1)
moving_average

array([1., 2., 3., 4.])

To adjust the step size of the sliding window, index the output view along
the desired dimension(s). Using the array shown above:


In [ ]:
v[::2]

array([[0, 1, 2],
       [2, 3, 4]])

You can slide in the reverse direction using the same technique:


In [ ]:
v[::-1]

array([[3, 4, 5],
       [2, 3, 4],
       [1, 2, 3],
       [0, 1, 2]])

The two examples below demonstrate the effect of ``writeable=True``.

Creating a view with the default ``writeable=False`` and then writing to
it raises an error.


In [ ]:
v = sliding_window_view(x, 3)
v[0,1] = 10


Traceback (most recent call last):
ValueError: assignment destination is read-only

Creating a view with ``writeable=True`` and then writing to it changes
the original array and multiple view positions.


In [ ]:
x = np.arange(6)  # reset x for the second example
v = sliding_window_view(x, 3, writeable=True)
v[0,1] = 10
x

array([ 0, 10,  2,  3,  4,  5])

In [ ]:
v

array([[ 0, 10,  2],
       [10,  2,  3],
       [ 2,  3,  4],
       [ 3,  4,  5]])

Note that a sliding window approach is often **not** optimal (see Notes).